<a href="https://colab.research.google.com/github/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/Processing/colab_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NER + DATE/TIME for the GeoRAG corpus
Upload `DATA.json` (or a JSON/JSONL file containing text records). The notebook runs the known BERT baseline for PER/ORG/LOC/MISC and spaCy for DATE/TIME. DATE/TIME are text mentions, not validated event dates.

Start with `LIMIT = 2`, inspect the outputs, then set `LIMIT = None` for the full corpus.


In [12]:
%pip -q install transformers spacy torch
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 37.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [13]:
import json
import subprocess
from pathlib import Path
import transformers
import spacy
import torch

הגדרת תיקיות

In [14]:
repo_dir = Path("/content/Corpus2GeoRAG_Pacific")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main",
         "https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git",
         str(repo_dir)],
        check=True,
    )

In [15]:
data_dir = repo_dir / "Data"
json_files = list(data_dir.glob("*.json"))
if len(json_files) != 1:
    raise ValueError(f"ציפיתי לקובץ JSON אחד בתיקיית Data; נמצאו: {[p.name for p in json_files]}")

קריאה לDATA

In [16]:
input_path = json_files[0]
output_dir = data_dir / "TMP_NER"

with input_path.open(encoding="utf-8") as f:
    data = json.load(f)

print(f"נטען: {input_path.name} — {len(data)} מסמכים")
print(f"פלט יישמר ב: {output_dir}")

נטען: DATA.json — 428 מסמכים
פלט יישמר ב: /content/Corpus2GeoRAG_Pacific/Data/TMP_NER


הצגת 3 הKEYS הראשונים

In [17]:
print(list(data.keys())[:3]) # שלוש הכותרות הראשונות

['14th Indian Infantry Division', '1939–1940 Winter Offensive', '1943 Cairo Declaration']


הגרלת 100 פסקאות

In [18]:
import json
import random
import re

SEED = 42
SAMPLE_SIZE = 100

paragraphs = []

for title, text in data.items():
    separators = list(re.finditer(r"\n[ \t]*\n+", text))
    starts = [0] + [m.end() for m in separators]
    ends = [m.start() for m in separators] + [len(text)]

    for index, (left, right) in enumerate(zip(starts, ends)):
        chunk = text[left:right]
        paragraph = chunk.strip()

        if not paragraph or re.fullmatch(r"=+\s*.*?\s*=+", paragraph):
            continue

        start = left + len(chunk) - len(chunk.lstrip())
        end = left + len(chunk.rstrip())

        paragraphs.append({
            "article_title": title,
            "paragraph_index": index,
            "start": start,
            "end": end,
            "text": text[start:end],
        })

paragraphs.sort(key=lambda p: (p["article_title"], p["start"]))

if SAMPLE_SIZE > len(paragraphs):
    raise ValueError(f"ביקשת {SAMPLE_SIZE} פסקאות, אך נמצאו רק {len(paragraphs)}")

sample = random.Random(SEED).sample(paragraphs, SAMPLE_SIZE)

output_dir.mkdir(parents=True, exist_ok=True)
sample_path = output_dir / f"sample_{SAMPLE_SIZE}_paragraphs_seed{SEED}.json"

result = {
    "source_file": input_path.name,
    "seed": SEED,
    "sample_size": SAMPLE_SIZE,
    "paragraphs": sample,
}

with sample_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"נשמרו {len(sample)} פסקאות מתוך {len(paragraphs)}")
print(sample_path)

נשמרו 100 פסקאות מתוך 9496
/content/Corpus2GeoRAG_Pacific/Data/TMP_NER/sample_100_paragraphs_seed42.json


העלאה לGIT

In [19]:
# import base64
# import requests
# from google.colab import userdata

# token = userdata.get("COLAB")
# if not token:
#     raise ValueError("לא נמצא GITHUB_TOKEN ב-Colab Secrets")

# repo_path = f"Data/TMP_NER/{sample_path.name}"
# url = f"https://api.github.com/repos/danielbr9382/Corpus2GeoRAG_Pacific/contents/{repo_path}"
# headers = {
#     "Authorization": f"Bearer {token}",
#     "Accept": "application/vnd.github+json",
# }
# content = sample_path.read_bytes()

# existing = requests.get(url, headers=headers, params={"ref": "main"}, timeout=30)
# if existing.status_code == 200:
#     remote_content = base64.b64decode(existing.json()["content"])
#     if remote_content != content:
#         raise FileExistsError("ב-GitHub כבר קיים קובץ בשם הזה עם תוכן שונה")
#     print("הקובץ הזה כבר נמצא ב-GitHub")
# elif existing.status_code == 404:
#     response = requests.put(
#         url,
#         headers=headers,
#         json={
#             "message": f"Add NER sample {sample_path.name}",
#             "content": base64.b64encode(content).decode("ascii"),
#             "branch": "main",
#         },
#         timeout=60,
#     )
#     response.raise_for_status()
#     print("הקובץ הועלה ל-GitHub")
# else:
#     existing.raise_for_status()

# print(f"https://github.com/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/{repo_path}")

In [20]:
# error = response.json()

# print("status:", response.status_code)
# print("message:", error.get("message"))
# print("required permissions:",
#       response.headers.get("X-Accepted-GitHub-Permissions"))
# print("rate limit remaining:",
#       response.headers.get("X-RateLimit-Remaining"))

In [21]:
from pathlib import Path
from shutil import copy2

sample_path = Path(sample_path)
output_dir = Path("/content/TMP_NER")
output_dir.mkdir(parents=True, exist_ok=True)

saved_path = output_dir / sample_path.name
if sample_path.resolve() != saved_path.resolve():
    copy2(sample_path, saved_path)

print(f"המדגם נשמר ב־Colab: {saved_path}")

המדגם נשמר ב־Colab: /content/TMP_NER/sample_100_paragraphs_seed42.json


בחינת מודלי NER

In [22]:
def load_sample(path):
    with path.open(encoding="utf-8") as file:
        return json.load(file)

sample_data = load_sample(sample_path)
sample = sample_data["paragraphs"]

print("מספר פסקאות:", len(sample))
print("קובץ:", sample_path)

מספר פסקאות: 100
קובץ: /content/Corpus2GeoRAG_Pacific/Data/TMP_NER/sample_100_paragraphs_seed42.json


In [23]:
def validate_sample(sample, source_data):
    if len(sample) != 100:
        raise ValueError(f"ציפינו ל־100 פסקאות, נמצאו {len(sample)}")

    for paragraph in sample:
        title = paragraph["article_title"]
        start = paragraph["start"]
        end = paragraph["end"]
        text = paragraph["text"]

        if source_data[title][start:end] != text:
            raise ValueError(f"היסטים לא תואמים: {title}, {start}:{end}")

    return True

validate_sample(sample, data)
print("כל 100 הפסקאות תואמות לטקסט המקורי")

כל 100 הפסקאות תואמות לטקסט המקורי


In [24]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch

MODEL_ID = "dslim/bert-base-NER"
REVISION = "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION, use_fast=True
)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID, revision=REVISION
)

ner = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="average",
    stride=64,
    device=0 if torch.cuda.is_available() else -1,
)

print("המודל נטען:", MODEL_ID)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


המודל נטען: dslim/bert-base-NER


In [25]:
import pandas as pd
# def extract_mentions(paragraph):
#     text = paragraph["text"]
#     predictions = ner(text)
#     mentions = []

#     for prediction in predictions:
#         start = int(prediction["start"])
#         end = int(prediction["end"])

#         mentions.append({
#             "text": text[start:end],
#             "label": prediction["entity_group"],
#             "start": start,
#             "end": end,
#             "start_document": paragraph["start"] + start,
#             "end_document": paragraph["start"] + end,
#             "score": float(prediction["score"]),
#         })

#     return mentions, predictions

# mentions_first, raw_first = extract_mentions(sample[0])
# pd.DataFrame(mentions_first)

def extract_mentions(paragraph, ner_pipeline=ner):
    text = paragraph["text"]
    predictions = ner_pipeline(text)
    mentions = []

    for prediction in predictions:
        start = int(prediction["start"])
        end = int(prediction["end"])

        mentions.append({
            "text": text[start:end],
            "label": prediction["entity_group"],
            "start": start,
            "end": end,
            "start_document": paragraph["start"] + start,
            "end_document": paragraph["start"] + end,
            "score": float(prediction["score"]),
        })

    return mentions, predictions
mentions_first, raw_first = extract_mentions(sample[0])
pd.DataFrame(mentions_first)

,text,label,start,end,start_document,end_document,score
0,HMS Prince of Wales,ORG,38,57,9052,9071,0.502382
1,HMS Repulse,ORG,77,88,9091,9102,0.787330
2,Singapore,LOC,130,139,9144,9153,0.999773
3,Japanese,MISC,173,181,9187,9195,0.999705
4,Malaya,LOC,210,216,9224,9230,0.999738
5,Japanese,MISC,218,226,9232,9240,0.999593
6,Malayan Peninsula,LOC,330,347,9344,9361,0.684488
7,Japanese,MISC,373,381,9387,9395,0.999658
8,Japanese,MISC,421,429,9435,9443,0.999623
9,Indian,MISC,494,500,9508,9514,0.999735


הרצה על כלל המדגם

In [28]:
bert_results = []

for paragraph in sample:
    mentions, raw = extract_mentions(paragraph)

    bert_results.append({
        "paragraph": paragraph,
        "mentions": mentions,
        "pipeline_output": raw,
    })

print("עובדו", len(bert_results), "פסקאות")

עובדו 100 פסקאות


In [29]:
import json
import uuid
from datetime import datetime, timezone

output_dir = saved_path.parent  # אותה תיקייה של המדגם ב-Colab
model_name = MODEL_ID.replace("/", "_")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id += "_" + uuid.uuid4().hex[:8]

ner_path = output_dir / f"ner_{model_name}_sample100_seed{sample_data['seed']}_{run_id}.json"

def json_number(value):
    if hasattr(value, "item"):
        return value.item()
    raise TypeError(f"Cannot save {type(value).__name__}")

output = {
    "model_id": MODEL_ID,
    "revision_model": REVISION,
    "sample_file": saved_path.name,
    "seed": sample_data["seed"],
    "results": bert_results,
}

with ner_path.open("x", encoding="utf-8") as file:
    json.dump(output, file, ensure_ascii=False, indent=2, default=json_number)

print(f"תוצאות ה־NER נשמרו ב־Colab: {ner_path}")

תוצאות ה־NER נשמרו ב־Colab: /content/TMP_NER/ner_dslim_bert-base-NER_sample100_seed42_20261006T154846Z_219bd7a8.json


מודל Jean-Baptiste/roberta-large-ner-english

טעינת המודל

In [ ]:
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch

ROBERTA_MODEL_ID = "Jean-Baptiste/roberta-large-ner-english"
ROBERTA_REVISION = model_info(ROBERTA_MODEL_ID).sha

roberta_tokenizer = AutoTokenizer.from_pretrained(
    ROBERTA_MODEL_ID, revision=ROBERTA_REVISION, use_fast=True
)
roberta_model = AutoModelForTokenClassification.from_pretrained(
    ROBERTA_MODEL_ID, revision=ROBERTA_REVISION
)

roberta_ner = pipeline(
    "ner",
    model=roberta_model,
    tokenizer=roberta_tokenizer,
    aggregation_strategy="average",
    stride=64,
    device=0 if torch.cuda.is_available() else -1,
)

print("המודל נטען:", ROBERTA_MODEL_ID)

בדיקה על פסקה אחת

In [ ]:
import pandas as pd

roberta_mentions_first, roberta_raw_first = extract_mentions(
    sample[0], roberta_ner
)

pd.DataFrame(roberta_mentions_first)

הרצה על כל הפסקאות

In [ ]:
roberta_results = []

for number, paragraph in enumerate(sample, start=1):
    mentions, raw = extract_mentions(paragraph, roberta_ner)

    roberta_results.append({
        "paragraph": paragraph,
        "mentions": mentions,
        "pipeline_output": raw,
    })

    if number % 10 == 0:
        print(f"עובדו {number}/{len(sample)} פסקאות")

print("מספר פסקאות בתוצאות:", len(roberta_results))

שמירת המדגם

In [ ]:
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

output_dir = Path("/content/TMP_NER")
local_sample_path = output_dir / Path(sample_path).name

if not local_sample_path.is_file():
    raise FileNotFoundError(f"קובץ המדגם לא נמצא: {local_sample_path}")

if len(roberta_results) != sample_data["sample_size"]:
    raise ValueError("ההרצה לא הסתיימה על כל המדגם")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id += "_" + uuid.uuid4().hex[:8]

model_name = ROBERTA_MODEL_ID.replace("/", "_")
result_path = output_dir / (
    f"ner_{model_name}_{local_sample_path.stem}_{run_id}.json"
)

def json_number(value):
    if hasattr(value, "item"):
        return value.item()
    raise TypeError(f"Cannot save {type(value).__name__}")

output = {
    "model_id": ROBERTA_MODEL_ID,
    "revision_model": ROBERTA_REVISION,
    "sample_file": local_sample_path.name,
    "seed": sample_data["seed"],
    "aggregation_strategy": "average",
    "stride_tokens": 64,
    "results": roberta_results,
}

with result_path.open("x", encoding="utf-8") as file:
    json.dump(output, file, ensure_ascii=False, indent=2,
              default=json_number)

print("התוצאות נשמרו:", result_path)